In [4]:
!pip uninstall -y torch torchaudio speechbrain \
                  pyannote.audio pyannote.core pyannote.metrics \
                  pytorch-lightning torchmetrics huggingface_hub -q

!pip install -q \
    torch==2.3.1+cu121 \
    torchaudio==2.3.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

# huggingface_hub 0.23.x is the last version where use_auth_token still works
!pip install -q "huggingface_hub==0.23.4"
!pip install -q "speechbrain==1.0.1"
!pip install -q "pyannote.metrics==3.2.1"
!pip install -q "pyannote.core==5.0.0"
!pip install -q openai-whisper
!pip install -q soundfile
!pip install -q scikit-learn

print("\n✅ Done. Go to  Runtime > Restart session  then run from Cell ")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lightning 2.6.1 requires pytorch-lightning, which is not installed.
lightning 2.6.1 requires torchmetrics<3.0,>0.7.0, which is not installed.
accelerate 1.12.0 requires huggingface_hub>=0.21.0, which is not installed.
sentence-transformers 5.2.3 requires huggingface-hub>=0.20.0, which is not installed.
peft 0.18.1 requires huggingface_hub>=0.25.0, which is not installed.
torchtune 0.6.1 requires huggingface_hub[hf_transfer], which is not installed.
timm 1.0.25 requires huggingface_hub, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0

In [1]:
from google.colab import files
import os

print("Upload your meeting audio file (.wav or .mp3):")
uploaded = files.upload()

RAW_AUDIO_PATH = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {RAW_AUDIO_PATH}")
print(f"   Size    : {os.path.getsize(RAW_AUDIO_PATH)/1024/1024:.1f} MB")


Upload your meeting audio file (.wav or .mp3):


Saving meeting1.wav to meeting1 (3).wav

✅ Uploaded: meeting1 (3).wav
   Size    : 19.4 MB


In [2]:
import torch
import torchaudio
import wave

waveform, orig_sr = torchaudio.load(RAW_AUDIO_PATH)
print(f"Original  : shape={waveform.shape}  sr={orig_sr}")

# Convert to mono
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# Resample to 16kHz
TARGET_SR = 16000
if orig_sr != TARGET_SR:
    resampler = torchaudio.transforms.Resample(orig_sr, TARGET_SR)
    waveform  = resampler(waveform)

MEETING_ID     = os.path.splitext(os.path.basename(RAW_AUDIO_PATH))[0]
PROCESSED_PATH = f"{MEETING_ID}_16k.wav"
torchaudio.save(PROCESSED_PATH, waveform, TARGET_SR)

# Print audio info
duration = waveform.shape[1] / TARGET_SR
print(f"Processed : shape={waveform.shape}  sr={TARGET_SR}")
print(f"Duration  : {duration:.1f}s  ({int(duration//60)}m {int(duration%60)}s)")
print(f"Saved     : {PROCESSED_PATH}")



Original  : shape=torch.Size([1, 20362240])  sr=16000
Processed : shape=torch.Size([1, 20362240])  sr=16000
Duration  : 1272.6s  (21m 12s)
Saved     : meeting1 (3)_16k.wav


In [8]:

import numpy as np
from speechbrain.inference.speaker import EncoderClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print("Loading SpeechBrain ECAPA-TDNN encoder ...")
encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb",
    run_opts={"device": device},
)
encoder.eval()
print("✅ Encoder loaded")

# Sliding window over audio
WINDOW_S    = 500
HOP_S       = 250
win_samples = int(WINDOW_S * TARGET_SR)
hop_samples = int(HOP_S * TARGET_SR)
waveform_np = waveform.squeeze().numpy()

windows      = []
window_times = []
i = 0
while i + win_samples <= len(waveform_np):
    windows.append(waveform_np[i:i + win_samples])
    window_times.append({
        "start": round(i / TARGET_SR, 3),
        "end":   round((i + win_samples) / TARGET_SR, 3),
    })
    i += hop_samples

print(f"Extracted {len(windows)} windows of {WINDOW_S}s")

# Compute embeddings
print("Computing embeddings ...")
embeddings = []
with torch.no_grad():
    for chunk in windows:
        tensor = torch.tensor(chunk).unsqueeze(0).float().to(device)
        emb    = encoder.encode_batch(tensor)
        embeddings.append(emb.squeeze().cpu().numpy())

embeddings = np.array(embeddings)
print(f"✅ Embeddings shape: {embeddings.shape}")


Device: cpu
Loading SpeechBrain ECAPA-TDNN encoder ...
✅ Encoder loaded
Extracted 4 windows of 500s
Computing embeddings ...
✅ Embeddings shape: (4, 192)


In [10]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

emb_norm = normalize(embeddings)

# ── Set number of speakers ───────────────────────────────────────
# AMI meetings = 4, your own meeting = however many speakers
NUM_SPEAKERS = 4   # ← change to match your audio

km     = KMeans(n_clusters=NUM_SPEAKERS, random_state=42, n_init=10)
labels = km.fit_predict(emb_norm)
print(f"✅ {NUM_SPEAKERS} speakers clustered across {len(labels)} windows")


✅ 4 speakers clustered across 4 windows


In [11]:
from collections import defaultdict

# Smooth labels with majority vote
def smooth_labels(labels, window=5):
    smoothed = labels.copy()
    for i in range(len(labels)):
        s = max(0, i - window // 2)
        e = min(len(labels), i + window // 2 + 1)
        smoothed[i] = np.bincount(labels[s:e]).argmax()
    return smoothed

labels_smooth = smooth_labels(labels, window=5)

# Build segments
hyp_segments = []
seg_start = window_times[0]["start"]
seg_label = labels_smooth[0]

for i in range(1, len(window_times)):
    if labels_smooth[i] != seg_label:
        hyp_segments.append({
            "speaker": f"speaker{seg_label+1}",
            "start":   seg_start,
            "end":     window_times[i-1]["end"],
        })
        seg_start = window_times[i]["start"]
        seg_label = labels_smooth[i]

hyp_segments.append({
    "speaker": f"speaker{seg_label+1}",
    "start":   seg_start,
    "end":     window_times[-1]["end"],
})

# Merge short segments < 0.5s
merged = []
for seg in hyp_segments:
    if seg["end"] - seg["start"] < 0.5 and merged:
        merged[-1]["end"] = seg["end"]
    else:
        merged.append(dict(seg))
hyp_segments = merged

# ── Speaker duration summary table ──────────────────────────────
spk_dur   = defaultdict(float)
for seg in hyp_segments:
    spk_dur[seg["speaker"]] += seg["end"] - seg["start"]
total_dur = sum(spk_dur.values()) or 1

print("\n" + "=" * 58)
print(f"  {'SPEAKER':<12} {'DURATION':>12}   {'SHARE':>7}   BAR")
print("=" * 58)
for spk, dur in sorted(spk_dur.items()):
    mins = int(dur // 60)
    secs = dur % 60
    pct  = 100 * dur / total_dur
    bar  = "█" * int(pct / 2)
    print(f"  {spk:<12} {mins:02d}m {secs:05.2f}s   {pct:6.1f}%   {bar}")
print("=" * 58)
print(f"  {'TOTAL':<12} {int(total_dur//60):02d}m {total_dur%60:05.2f}s   100.0%")
print("=" * 58)
print(f"\n  Total turns : {len(hyp_segments)}")

# ── Turn-by-turn timeline ────────────────────────────────────────
print("\n" + "=" * 65)
print("  TURN-BY-TURN TIMELINE")
print("=" * 65)
for seg in hyp_segments:
    mm_s = int(seg["start"] // 60); ss_s = seg["start"] % 60
    mm_e = int(seg["end"]   // 60); ss_e = seg["end"]   % 60
    dur  = seg["end"] - seg["start"]
    print(f"  [{mm_s:02d}:{ss_s:05.2f} --> {mm_e:02d}:{ss_e:05.2f}]  "
          f"{seg['speaker']:<12}  ({dur:.1f}s)")
print("=" * 65)




  SPEAKER          DURATION     SHARE   BAR
  speaker1     20m 50.00s    100.0%   ██████████████████████████████████████████████████
  TOTAL        20m 50.00s   100.0%

  Total turns : 1

  TURN-BY-TURN TIMELINE
  [00:00.00 --> 20:50.00]  speaker1      (1250.0s)


In [12]:
import whisper

whisper_model = whisper.load_model("base")

print(f"Audio: {duration:.1f}s = {int(duration//60)}m {int(duration%60)}s")
print("Running Whisper STT...")
result = whisper_model.transcribe(PROCESSED_PATH, word_timestamps=True)

stt_words = []
for seg in result["segments"]:
    for w in seg.get("words", []):
        stt_words.append({
            "word":  w["word"].strip(),
            "start": round(w["start"], 3),
            "end":   round(w["end"],   3),
        })

print(f"STT: {len(result['segments'])} segments")
print(f"\nPreview: {result['text'][:120]}...")

# Print segment table (matches your screenshot)
print(f"\n{'#':<4} {'Start':>7} {'End':>7}  {'Text'}")
print("-" * 60)
for i, seg in enumerate(result["segments"]):
    start = seg["start"]; end = seg["end"]
    text  = seg["text"].strip()[:45]
    print(f"{i:<4} {start:>5.2f}s  {end:>5.2f}s  {text}")

print(f"\n{'▬'*60}")
print(f"⚠  Note the segment numbers ↑")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 175MiB/s]


Audio: 1272.6s = 21m 12s
Running Whisper STT...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


STT: 238 segments

Preview:  it's back and it's going to be the link of our previous level now. And this is what we're going to be doing over the ne...

#      Start     End  Text
------------------------------------------------------------
0    39.84s  59.98s  it's back and it's going to be the link of ou
1    63.24s  66.64s  And this is what we're going to be doing over
2    68.30s  72.34s  So, first of all, just to make sure that we a
3    72.90s  74.34s  And we're at the Project Manager.
4    74.82s  77.28s  So, I'm just introducing you to that today.
5    77.80s  80.38s  I'm doing it and I'm especially interested in
6    80.62s  81.04s  Okay.
7    81.94s  86.40s  I'm Andrew and I'm marketing experts.
8    87.26s  88.44s  I'm creating a project here.
9    92.34s  94.26s  So, we're designing a new remote control.
10   96.20s  97.76s  I have to record a piece of it.
11   98.70s  101.10s  So, do that under the environment.
12   104.00s  111.70s  And then we're designing a remote contr

In [13]:
def merge_stt_diarization(hyp_segments, stt_words):
    transcript = []
    for seg in hyp_segments:
        words_in_seg = [
            w["word"] for w in stt_words
            if w["start"] >= seg["start"] and w["end"] <= seg["end"]
        ]
        transcript.append({
            "speaker": seg["speaker"],
            "start":   seg["start"],
            "end":     seg["end"],
            "text":    " ".join(words_in_seg).strip() or "[inaudible]",
        })
    return transcript


transcript = merge_stt_diarization(hyp_segments, stt_words)

# Print in same style as your screenshot
print("\n" + "=" * 60)
for seg in transcript:
    if seg["text"] == "[inaudible]":
        continue
    print(f"\n{seg['speaker']}:")
    print(f"  {seg['text']}")
print("\n" + "=" * 60)



speaker1:
  it's back and it's going to be the link of our previous level now. And this is what we're going to be doing over the next 25 minutes. So, first of all, just to make sure that we all know each other. And we're at the Project Manager. So, I'm just introducing you to that today. I'm doing it and I'm especially interested in this. Okay. I'm Andrew and I'm marketing experts. I'm creating a project here. So, we're designing a new remote control. I have to record a piece of it. So, do that under the environment. And then we're designing a remote control. So, this is the spiritual, trendy and new family. So, that's kind of where I'll be. And so, there are three different stages to the design. I know you share what you guys have already seen. Can you make a list of pictures of them? I just found that they're ready to answer. Yeah, it's... So, we're going to have like individual work and then we're going to make it. Repeat the process. Repeat the process. And we just put a little b

In [14]:
import json
from google.colab import files

# Human-readable transcript .txt
txt_path = f"{MEETING_ID}_transcript.txt"
with open(txt_path, "w") as f:
    f.write(f"Meeting : {MEETING_ID}\n")
    f.write("=" * 65 + "\n\n")
    for seg in transcript:
        if seg["text"] == "[inaudible]":
            continue
        f.write(f"{seg['speaker']}:\n")
        f.write(f"  [{int(seg['start']//60):02d}:{seg['start']%60:05.2f} --> "
                f"{int(seg['end']//60):02d}:{seg['end']%60:05.2f}]\n")
        f.write(f"  {seg['text']}\n\n")

output_files = {
    f"{MEETING_ID}_diarization.json": hyp_segments,
    f"{MEETING_ID}_transcript.json":  transcript,
}

for fname, data in output_files.items():
    with open(fname, "w") as f:
        json.dump(data, f, indent=2)

print("Downloading ...\n")
for fname in list(output_files.keys()) + [txt_path]:
    files.download(fname)
    print(f"  ⬇  {fname}")

print("\n✅ All done!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇  meeting1 (3)_diarization.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇  meeting1 (3)_transcript.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇  meeting1 (3)_transcript.txt

✅ All done!
